# Retention Explanation Stability

**Checkpoint 54 supporting notebook**  
**Synthetic-data notice:** every employee, outcome, score, and explanation in this notebook is fictional.

This notebook presents aggregate explanations for the selected Version 2 Logistic Regression and tests whether feature importance remains similar across employee-grouped refits. It does not save employee-level explanations, reopen the once-only final test, retune the frozen policy, or authorize an employment action.

## 1. Explanation contract

For transformed feature `j`, the exact interventional linear SHAP value is:

`coefficient_j × (transformed_value_j - historical_background_mean_j)`

The values explain the base Logistic Regression **log-odds**, not calibrated probability percentage points. One-hot contributions are grouped back to the 21 raw policy features.

In [1]:
from pathlib import Path
import pandas as pd
import yaml

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = project_root / "data" / "processed" / "retention_explanations"
with (project_root / "config" / "retention_explanations.yaml").open(encoding="utf-8") as handle:
    policy = yaml.safe_load(handle)

contract = policy["model_contract"]
print(f"Selected base model: {contract['selected_base_model']}")
print(f"Selected calibration: {contract['selected_calibration_method']}")
print(f"Historical rows: {contract['expected_historical_rows']:,}")
print(f"Current eligible rows: {contract['expected_current_rows']:,}")
print(f"Encoded features: {contract['expected_encoded_features']}")
print(f"Raw feature groups: {contract['expected_raw_features']}")
print("Employee-level explanation files: 0")

Selected base model: Logistic Regression
Selected calibration: Sigmoid
Historical rows: 16,368
Current eligible rows: 7,305
Encoded features: 35
Raw feature groups: 21
Employee-level explanation files: 0


## 2. Current aggregate model drivers

Mean absolute SHAP magnitude measures how strongly a raw feature changes current model scores in either direction. It does not show causation.

In [2]:
importance = pd.read_csv(
    output_dir / "retention_explanation_global_importance.csv"
)
columns = [
    "importance_rank",
    "raw_feature",
    "mean_absolute_shap_log_odds",
    "importance_share",
]
print(importance.head(10)[columns].round(4).to_string(index=False))

 importance_rank                  raw_feature  mean_absolute_shap_log_odds  importance_share
               1      salary_position_percent                       0.4586            0.2513
               2                 tenure_years                       0.2604            0.1427
               3           performance_rating                       0.2466            0.1352
               4              no_prior_review                       0.1268            0.0695
               5              department_name                       0.1064            0.0583
               6                    job_level                       0.1053            0.0577
               7           no_prior_promotion                       0.0930            0.0510
               8 completed_training_hours_12m                       0.0834            0.0457
               9    salary_growth_12m_percent                       0.0638            0.0350
              10            performance_trend                       0.

Salary position, tenure, and performance rating are the three largest aggregate model-score drivers. Their importance reflects the synthetic generator and fitted model; it is not evidence that changing any one feature would cause retention.

## 3. Employee-grouped explanation stability

Five diagnostic models are fitted with complete employee histories kept together. Each model explains the same current population. Pairwise Spearman correlation compares all 21 importance ranks, while top-10 Jaccard compares the leading feature sets.

In [3]:
pairwise = pd.read_csv(
    output_dir / "retention_explanation_pairwise_stability.csv"
)
summary = pd.DataFrame(
    {
        "metric": [
            "Minimum pairwise Spearman",
            "Median pairwise Spearman",
            "Minimum top-10 Jaccard",
        ],
        "value": [
            pairwise["spearman_rank_correlation"].min(),
            pairwise["spearman_rank_correlation"].median(),
            pairwise["top_10_jaccard"].min(),
        ],
    }
)
print(summary.round(4).to_string(index=False))

                   metric  value
Minimum pairwise Spearman 0.8857
 Median pairwise Spearman 0.9299
   Minimum top-10 Jaccard 0.6667


In [4]:
stability = pd.read_csv(
    output_dir / "retention_explanation_stability_summary.csv"
)
columns = [
    "raw_feature",
    "global_importance_rank",
    "minimum_fold_rank",
    "maximum_fold_rank",
    "top_k_fold_count",
    "direction_stability",
]
print(stability.head(10)[columns].to_string(index=False))

                 raw_feature  global_importance_rank  minimum_fold_rank  maximum_fold_rank  top_k_fold_count  direction_stability
     salary_position_percent                       1                  1                  1                 5                  1.0
                tenure_years                       2                  2                  3                 5                  1.0
          performance_rating                       3                  2                  3                 5                  1.0
             no_prior_review                       4                  4                  6                 5                  1.0
             department_name                       5                  4                  8                 5                  1.0
                   job_level                       6                  4                  8                 5                  1.0
          no_prior_promotion                       7                  5                  9

The broad importance pattern is stable, but exact ranks can move. A minimum top-10 Jaccard value of 0.6667 means the least-similar pair shares eight features out of a twelve-feature union. It does not mean every top-ten position is identical.

## 4. Validation and governance

In [5]:
validation = pd.read_csv(
    output_dir / "retention_explanation_validation.csv"
)
passed = int(validation["status"].eq("PASS").sum())
print(f"Validation checks passed: {passed}/{len(validation)}")
print("Employee-level explanations saved: 0")
print("Final-test or policy changes made: 0")
print(f"Human review required: {policy['governance']['human_review_required']}")
print(
    "Automatic employment action permitted: "
    f"{policy['governance']['automatic_employment_action_permitted']}"
)

Validation checks passed: 21/21
Employee-level explanations saved: 0
Final-test or policy changes made: 0
Human review required: True
Automatic employment action permitted: False


## 5. Interpretation limits

- These explanations describe model associations, not reasons an employee will leave.
- Log-odds contributions are not calibrated probability percentage points.
- The interventional background assumption does not fully represent correlated features.
- Grouped stability does not guarantee future stability under data drift.
- Synthetic relationships do not establish validity for a real employer.
- Scores and explanations support structured human review only; automatic employment action is prohibited.

See `docs/model_explanations_and_stability.md` for the complete method, outputs, validation rules, and limitations.